# Regresión Logística para diagnóstico de Rickettsiosis

Esthefania Ortega

## 0. Carga de librerías, dependencias y datos

In [ ]:
import pandas as pd
import dagshub
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import recall_score, make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
import seaborn as sns
import matplotlib.pyplot as plt

rick_sthef = pd.read_csv('../data/interim/rick_pca.csv')


## 0.1. Configuración MLOps

In [ ]:
dagshub.init(repo_owner='Mario-ACL', repo_name='ml_project', mlflow=True)

## 1. Preprocesamiento para modelado

### Filtrado a solo casos confirmados y descartados

In [ ]:
rick_clases = rick_sthef[rick_sthef['estatus_caso'].isin([2, 3])].copy()


Mapeamos a 0 y 1, donde:

- 0 es caso descartado
- 1 es caso confirmado

In [ ]:
map_estatus = {2: 1, # Casos confirmados
               3: 0} # Casos descartados

rick_clases = (
    rick_sthef[rick_sthef['estatus_caso'].isin([2, 3])]
    .copy()
    .assign(estatus_caso=lambda df: df['estatus_caso'].map(map_estatus))
)

In [ ]:
X = rick_clases.drop(columns = ['estatus_caso', 'embarazo', 'sem_gest', 'es_indigena']).copy()
y = rick_clases['estatus_caso'].copy()

In [ ]:
# Identificar automáticamente columnas binarias codificadas como 1/2
def recodificar_binarias(df):
    df_rec = df.copy()
    cols_recodificadas = []
    
    for col in df_rec.columns:
        valores_unicos = set(df_rec[col].dropna().unique())
        if valores_unicos.issubset({1, 2}):
            df_rec[col] = (df_rec[col] == 1).astype(int)
            cols_recodificadas.append(col)
    
    print(f"Columnas recodificadas (1/2 → 0/1): {len(cols_recodificadas)}")
    print(cols_recodificadas)
    return df_rec

X_rec = recodificar_binarias(X)

# Verificación
print("\nAntes:", rick_clases['hepatomegalia'].value_counts().to_dict())
print("Después:", X_rec['hepatomegalia'].value_counts().to_dict())

### Distribución de clases

In [ ]:
y.value_counts()

In [ ]:
print(f"Shape de X : {X_rec.shape}")
print(f"Shape de y : {y.shape}")
print(f"\nDistribución de clases:\n{y.value_counts()}")

In [ ]:
selector = VarianceThreshold(threshold=0.04)
selector.fit(X_rec)

vars_eliminadas = X_rec.columns[~selector.get_support()].tolist()
print(f"Variables eliminadas por baja varianza ({len(vars_eliminadas)}):")
print(vars_eliminadas)

X_filtrado = pd.DataFrame(
    selector.transform(X_rec),
    columns=X_rec.columns[selector.get_support()]
)

print(f"\nVariables antes  : {X_rec.shape[1]}")
print(f"Variables después: {X_filtered.shape[1]}")

In [ ]:
X_filtrado.columns

### Estandarización de datos

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_filtrado)

## 2. Modelado para selección de variables

In [ ]:
lr = LogisticRegression(l1_ratio = 1, 
                        solver='liblinear',
                        C = 1.0,
                        class_weight='balanced',
                        max_iter=1000,
                        random_state=42)
lr.fit(X_scaled, y)
accuracy = lr.score(X_scaled, y)
print(accuracy)

In [ ]:
coefs = lr.coef_[0]
n_seleccionadas = (coefs != 0).sum()
print(f"Variables seleccionadas por L1: {n_seleccionadas} de {len(coefs)}")

Probando diferentes niveles de C y evaluando con AUC-ROC y Recall:

In [ ]:
valores_C = [0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.1, 0.5, 1.0]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados_recall = []
for C in valores_C:
    m = LogisticRegression(
        l1_ratio=1, solver='liblinear',
        C=C, class_weight='balanced',
        max_iter=1000, random_state=42
    )
    
    scores_auc    = cross_val_score(m, X_scaled, y, cv=cv, scoring='roc_auc')
    scores_recall = cross_val_score(m, X_scaled, y, cv=cv, scoring='recall')
    
    m.fit(X_scaled, y)
    n_vars = (m.coef_[0] != 0).sum()
    
    resultados_recall.append({
        'C'           : C,
        'n_variables' : n_vars,
        'auc_media'   : scores_auc.mean(),
        'recall_media': scores_recall.mean(),
        'recall_std'  : scores_recall.std()
    })
    print(f"C={C:<6} | Variables: {n_vars:>2} | AUC: {scores_auc.mean():.3f} | Recall: {scores_recall.mean():.3f} ± {scores_recall.std():.3f}")

df_recall = pd.DataFrame(resultados_recall)

Analicemos las variables seleccionadas de los modelos con mayor Recall, nuestra métrica principal.

In [ ]:
for C in [0.02, 0.03]:
    m = LogisticRegression(
        l1_ratio=1, solver='liblinear',
        C=C, class_weight='balanced',
        max_iter=1000, random_state=42
    )
    m.fit(X_scaled, y)
    
    df_coefs = pd.DataFrame({
        'variable': X_filtered.columns,
        'coeficiente': m.coef_[0]
    }).query('coeficiente != 0').sort_values('coeficiente', key=abs, ascending=False)
    
    print(f"\nC={C} — {len(df_coefs)} variables:")
    print(df_coefs[['variable', 'coeficiente']].to_string(index=False))
    print("-"*50)

Los tres síntomas iniciales (llamados la triada) son: 
- Temperatura > 38°C
- Cefalea intensa
- Malestar general / mialgias o artralgias. 

Los primera no se encuentra dentro de las variables seleccionadas por ninguno de los dos modelos Lasso, y las últimas dos solo se encuentran en el segundo modelo analizaremos por qué. 

In [ ]:
# Ver coeficientes de las variables ausentes
vars_esperadas = ['temperatura', 'cefalea', 'mialgias']

for C in [0.03, 0.04]:
    m = LogisticRegression(
        l1_ratio=1, solver='liblinear',
        C=C, class_weight='balanced',
        max_iter=1000, random_state=42
    )
    m.fit(X_scaled, y)
    coefs = pd.Series(m.coef_[0], index=X_filtered.columns)
    print(f"\nC={C}:")
    for v in vars_esperadas:
        if v in coefs.index:
            print(f"  {v}: {coefs[v]:.6f}")
        else:
            print(f"  {v}: NO ESTÁ EN X_filtered")

In [ ]:
vars_esperadas = ['temperatura', 'cefalea', 'mialgias']

for v in vars_esperadas:
    print(f"\n{v}:")
    print(rick_clases.groupby('estatus_caso')[v].mean().rename({0: 'Descartados', 1: 'Confirmados'}))

Las variables de sintomas iniciales — fiebre, cefalea y mialgias — no resultan muy discriminativas entre casos confirmados y descartados, consistente con su rol como criterios de sospecha clínica que determinan el ingreso de un paciente. Las variables con mayor poder discriminativo fueron petequias, exantema, choque y contacto con otros animales.

Si consideramos las 7 variables de diferencia entre los modelos de C=0.02 y C = 0.03:
- 'ide_sex'
- 'ide_eda_ano'
- 'cefalea'
- 'mialgias'
- 'torniquete_positivo'
- 'faringitis'
- 'alteraciones_conciencias'

Cefalea y mialgias son de los principales signos, mencionadas en la Guía de Práctica Clínica (/references/GPC_FMRR_2023.pdf) de Rickettsiosis, así como alteraciones de conciencia son un signo poco común. Los sintomas de faringitis y torniquete_positivo funcionan como descartadores para enfermedades como dengue.

En cuanto a las variables demográficas pueden deberse a un sesgo, pero también es importante considerar que las ocupaciones que se desempeñan en su mayoría por hombres como el trabajo agrícola, ganadero y de construcción tienen riesgo a mayor exposición, mientras que en cuestión de edades, los menores de edad tienen mayor exposición. Evaluaremos a continuación. 

In [ ]:
for sexo, label in [(1, 'Hombres'), (2, 'Mujeres')]:
    mask = rick_clases['ide_sex'] == sexo
    X_grupo = X_scaled[mask]
    y_grupo = y[mask]
    y_pred_proba = modelo_final.predict_proba(X_grupo)[:, 1]
    auc = roc_auc_score(y_grupo, y_pred_proba)
    print(f"{label}: AUC = {auc:.3f}")

Confirmando que el sesgo es mínimo con la variable de sexo, elegimos la selección de variables generadas con el modelo utilizando C=0.03 que mantiene 29 variables de relevancia clínica.

In [ ]:
# Entrenar modelo final con C=0.03
modelo_final = LogisticRegression(
    l1_ratio=1, solver='liblinear',
    C=0.03, class_weight='balanced',
    max_iter=1000, random_state=42
)
modelo_final.fit(X_scaled, y)

# DataFrame de coeficientes
df_coefs_final = pd.DataFrame({
    'variable': X_filtered.columns,
    'coeficiente': modelo_final.coef_[0]
}).query('coeficiente != 0').sort_values('coeficiente', ascending=True)

# Gráfica
fig, ax = plt.subplots(figsize=(10, 9))

colores = ['#e53935' if c > 0 else '#1e88e5' for c in df_coefs_final['coeficiente']]

sns.barplot(
    data=df_coefs_final,
    x='coeficiente',
    y='variable',
    palette=colores,
    ax=ax
)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente  (rojo = aumenta probabilidad de rickettsiosis, azul = la reduce)')
ax.set_ylabel('')
ax.set_title('Variables seleccionadas — Regresión Logística L1 (C=0.04)', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

**Variable predictoras positivas:**
Las tres variables principales que aumentan la probabilidad de Rickettsiosis son: exantema, choque, petequias. Concuerdan con los signos principales para diferenciar la enfermedad de otras febriles, sin embargo, son signos que aparecen una vez que la enfermedad ha evolucionado en adultos, disminuyendo la ventana de oportunidad para un tratamiento oportuno. En casos pediátricos sí funciona como un predictor ya que suelen aparecer en las primeras 48 horas 

Observando las siguientes tres: contacto con otro animal (diferente a garrapatas), dolor abdominal intenso y taquicardia, sib síntomas de compromiso sistémico progresivo.

**Variable predictoras negativas:**
Las tres variables principales que apuntan hacia un diagnóstico negativo son: prurito, tos y dolor retroocular. Estas variables pueden apuntar hacia enfermedades de infección respiratoria.  



El modelo aprendió no solo qué síntomas confirman rickettsiosis, sino cuáles la descartan activamente y esos síntomas negativos apuntan coherentemente hacia el diagnóstico diferencial principal en Sonora: dengue e infecciones respiratorias

## 3. Evaluación del modelo

In [ ]:
with mlflow.start_run(run_name="logreg-featselection"):
    c = 0.03
    penalty = 'l1'
    solver = 'liblinear'
    class_weight = 'balanced'
    max_iter = 1000
    rs = 42

    modelo_final = LogisticRegression(
        l1_ratio=1, solver=solver,
        C=c, class_weight=class_weight,
        max_iter=max_iter, random_state=rs
    )
    modelo_final.fit(X_scaled, y)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=rs)
    scoring = {
        'roc_auc': 'roc_auc',
        'recall': 'recall',
        'especificidad': make_scorer(recall_score, pos_label=0),
        'f1': 'f1',
    }
    scores = cross_validate(
        modelo_final, X_scaled, y,
        cv=cv, scoring=scoring,
        return_train_score=False, n_jobs=-1
    )

    metricas = {
        'AUC-ROC': scores['test_roc_auc'],
        'Recall': scores['test_recall'],
        'Especificidad': scores['test_especificidad'],
        'F1-Score': scores['test_f1'],
    }

    print(f"Evaluación — Regresión Logística L1 (C={c})")
    print(f"{'='*50}")
    for metrica, valores in metricas.items():
        print(f"{metrica:<15}: {valores.mean():.3f} ± {valores.std():.3f}")

    mlflow.log_params({
        'C': c,
        'penalty': penalty,
        'solver': solver,
        'class_weight': class_weight,
        'max_iter': max_iter,
        'random_state': rs
    })
    mlflow.log_metric('auc-roc', metricas['AUC-ROC'].mean())
    mlflow.log_metric('recall', metricas['Recall'].mean())
    mlflow.log_metric('especificidad', metricas['Especificidad'].mean())
    mlflow.log_metric('f1-score', metricas['F1-Score'].mean())

In [ ]:
# Variables seleccionadas
vars_seleccionadas = X_filtrado.columns[modelo_final.coef_[0] != 0].tolist()

print(f"Variables seleccionadas: {len(vars_seleccionadas)}")
print(vars_seleccionadas)

# Crear dataframe con variables seleccionadas + target
df_modelo = rick_clases[vars_seleccionadas + ['estatus_caso']].copy()
df_modelo_rec = recodificar_binarias(df_modelo)

print(f"\nShape final: {df_modelo_rec.shape}")
df_modelo_rec.head()

## 4. Exportación de datos resultantes

In [ ]:
df_modelo_rec.to_csv('../data/processed/rick_feat_selected.csv', index=False)